    # Pregunta 1 — Anomalías de rodilla en MRI, paso a paso

    Un dataset de **4 407 estudios** de resonancia de rodilla donde solo
    **58** tienen etiquetas puestas por un radiólogo. Los otros 4 349 traen el
    informe en texto libre, en **nueve idiomas**. Hay que predecir **doce
    hallazgos a la vez**, y las etiquetas con las que se entrena hay que
    fabricarlas leyendo esos informes.

    El problema real de esta pregunta **no es la arquitectura**: es que la
    supervisión es ruidosa y hay que medir cuánto.


> **Cómo usar este cuaderno.** Las celdas de código son **seguras de correr**:
> inspeccionan los datos y los resultados que ya están en disco, pero
> **ninguna reentrena nada**. Los comandos que sí entrenan están escritos como
> texto, no como celdas, para que no se disparen por accidente al hacer
> "ejecutar todo".
>
> Si una celda dice que falta un archivo, es que ese paso todavía no corrió:
> el README del paso correspondiente tiene el comando exacto.

In [1]:
import json
import pathlib
import sys

# La raiz del proyecto se busca hacia arriba, asi el cuaderno funciona tanto
# si se abre desde la carpeta de la pregunta como desde la raiz.
# Raiz del proyecto en la maquina local. Orden: la variable REDES_RAIZ, la
# carpeta actual o alguna superior, y por ultimo la ruta fija de esta PC. Asi
# funciona en Jupyter/VS Code y en Colab conectado a un entorno de ejecucion
# local, aunque Jupyter se haya lanzado desde otra carpeta.
import os
RAIZ_LOCAL = pathlib.Path("D:/Backend/redesneuronales")
RAIZ = pathlib.Path(os.environ.get("REDES_RAIZ", pathlib.Path.cwd()))
while not (RAIZ / "pregunta1").is_dir() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
if not (RAIZ / "pregunta1").is_dir():
    RAIZ = RAIZ_LOCAL
if not (RAIZ / "pregunta1").is_dir():
    raise FileNotFoundError(
        f"No encuentro el proyecto (busque desde {pathlib.Path.cwd()} y en "
        f"{RAIZ_LOCAL}). En Colab: Conectar -> 'Conectar a un entorno de "
        "ejecucion local', o define REDES_RAIZ con la ruta del proyecto.")
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))
print("raiz del proyecto:", RAIZ)


def leer(ruta):
    """Lee un JSON de resultados; avisa con claridad si todavia no existe.

    Acepta una ruta relativa a la raiz o una absoluta (la que da rutas.py).
    """
    p = pathlib.Path(ruta)
    if not p.is_absolute():
        p = RAIZ / p
    if not p.is_file():
        print(f"  (falta {ruta}: ese paso todavia no corrio)")
        return None
    return json.loads(p.read_text(encoding="utf-8"))

raiz del proyecto: D:\Backend\redesneuronales


In [2]:
import csv

# Que hay en disco.
from pregunta1.rutas import TRAIN_CSV, describir
print(describir())
base = TRAIN_CSV
if base.is_file():
    with base.open(encoding="utf-8") as f:
        filas = list(csv.DictReader(f))
    cols = list(filas[0])
    etiquetas = [c for c in cols if c not in ("StudyInstanceUID", "Report")]
    dorados = [r for r in filas if any(r[c] not in ("", None) for c in etiquetas)]
    print(f"train.csv: {len(filas):,} filas | {len(etiquetas)} etiquetas")
    print(f"  con alguna etiqueta dorada: {len(dorados)}")
    print(f"  etiquetas: {etiquetas}")

datos de la Pregunta 1 en pregunta1/datos/
train.csv: 4,407 filas | 12 etiquetas
  con alguna etiqueta dorada: 58
  etiquetas: ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']


# Paso 1 — Etiquetas débiles (entregable 1)

Hay que sacar doce etiquetas de un informe de texto libre. La tentación es
buscar palabras clave, y no alcanza por una razón: **la mitad de las veces
que un informe nombra un hallazgo es para decir que NO está**.

> "There is no evidence of a meniscal tear."

Un buscador de palabras marca "meniscal tear" como presente. El extractor
tiene que entender la **negación**, y además la **incertidumbre**
("possible", "cannot be excluded") y el **antecedente** ("status post ACL
reconstruction" es una cirugía vieja, no una lesión nueva).

In [3]:
from pregunta1.paso1_etiquetas_debiles.extractor import ExtractorReglas

extractor = ExtractorReglas()
ejemplos = [
    "There is a complete tear of the anterior cruciate ligament.",
    "There is no evidence of a meniscal tear.",
    "Possible small joint effusion, cannot be excluded.",
    "Status post ACL reconstruction, graft intact.",
]
for texto in ejemplos:
    print(f"\n{texto}")
    r = extractor.extraer(texto)
    evid = getattr(r, "evidencias", [])
    if not evid:
        print("    (ninguna evidencia)")
    for e in evid:
        print(f"    {e.etiqueta:<20} {e.estado.name}")
print("\n  Mira la segunda y la tercera: el termino ESTA en el texto, pero")
print("  el estado no es afirmado. Un buscador de palabras clave las")
print("  marcaria positivas a las cuatro.")


There is a complete tear of the anterior cruciate ligament.
    ACL                  AFIRMADO

There is no evidence of a meniscal tear.
    (ninguna evidencia)

Possible small joint effusion, cannot be excluded.
    Effusion             INCIERTO

Status post ACL reconstruction, graft intact.
    (ninguna evidencia)

  Mira la segunda y la tercera: el termino ESTA en el texto, pero
  el estado no es afirmado. Un buscador de palabras clave las
  marcaria positivas a las cuatro.


## Cuánta supervisión se logra, y de qué calidad

El número que importa no es "cuántas etiquetas salieron" sino **cuántas
coinciden con el radiólogo**. Para eso están los 58 dorados.

In [4]:
import collections

ruta = RAIZ / "pregunta1/paso1_etiquetas_debiles/etiquetas_debiles.csv"
if ruta.is_file():
    with ruta.open(encoding="utf-8") as f:
        filas = list(csv.DictReader(f))
    META = {"StudyInstanceUID", "idioma", "traducido", "n_oraciones",
            "n_evidencias", "es_dorado", "veredicto"}
    ET = [c for c in filas[0] if c not in META]
    sin = [r for r in filas if all(r[c] in ("0", "") for c in ET)]
    print(f"{len(filas):,} estudios procesados")
    print(f"  sin ninguna etiqueta: {len(sin)} = {100*len(sin)/len(filas):.1f}%")
    print(f"  veredicto:", dict(collections.Counter(r["veredicto"] for r in filas)))
    print("\n  'ilegible' es el unico que significa 'no supe leerlo'.")
    print("  'sano_afirmado' y 'sano_negado' son informacion REAL: una")
    print("  rodilla sana. Confundirlos con un fallo del extractor fue el")
    print("  error que motivo agregar la columna 'veredicto'.")

4,407 estudios procesados
  sin ninguna etiqueta: 807 = 18.3%
  veredicto: {'con_hallazgo': 3600, 'ilegible': 196, 'sano_afirmado': 343, 'sano_negado': 268}

  'ilegible' es el unico que significa 'no supe leerlo'.
  'sano_afirmado' y 'sano_negado' son informacion REAL: una
  rodilla sana. Confundirlos con un fallo del extractor fue el
  error que motivo agregar la columna 'veredicto'.


## El sesgo lingüístico (entregable 7)

Este es el hallazgo más incómodo del paso, y es el que sostiene el
entregable 7.

In [5]:
if ruta.is_file():
    tot = collections.Counter(r["idioma"] for r in filas)
    vac = collections.Counter(r["idioma"] for r in sin)
    print(f"{'idioma':<8} {'estudios':>9} {'sin etiqueta':>13} {'%':>7}")
    print("-" * 42)
    for i, n in tot.most_common():
        print(f"{i:<8} {n:>9} {vac[i]:>13} {100*vac[i]/n:>6.1f}%")
    peor = max(tot, key=lambda i: vac[i]/tot[i])
    print(f"\n  El extractor falla {100*vac[peor]/tot[peor]/ (100*vac['en']/tot['en']):.1f} veces")
    print(f"  mas seguido en '{peor}' que en ingles.")
    print("\n  En un despliegue clinico real eso significa que el sistema")
    print("  sirve peor en los centros que no escriben en ingles, y el")
    print("  sesgo no se ve en una metrica agregada.")

idioma    estudios  sin etiqueta       %
------------------------------------------
en            1736           200   11.5%
es             682           129   18.9%
tr             546           137   25.1%
hr             406            38    9.4%
el             321           174   54.2%
de             262            70   26.7%
bg             220            24   10.9%
nl             153            27   17.6%
fr              81             8    9.9%

  El extractor falla 4.7 veces
  mas seguido en 'el' que en ingles.

  En un despliegue clinico real eso significa que el sistema
  sirve peor en los centros que no escriben en ingles, y el
  sesgo no se ve en una metrica agregada.


# Paso 2 — Preprocesamiento

El enunciado recomienda explícitamente **no procesar los 819 640 archivos
DICOM** (569 GB), sino trabajar con un subconjunto estratificado de 300 a
600 estudios, uno o dos planos, y cortes 2D o un volumen 2.5D pequeño.

El problema real de este paso es que **las imágenes no se parecen entre
sí**: distinta resolución, distinto campo de visión, distinto fabricante,
y más de la mitad sin el tag de lateralidad.

In [6]:
ruta = RAIZ / "pregunta1/paso2_preprocesamiento/estudios_seleccionados.csv"
if ruta.is_file():
    with ruta.open(encoding="utf-8") as f:
        sel = list(csv.DictReader(f))
    ET = [c for c in sel[0] if c not in
          ("StudyInstanceUID", "SeriesInstanceUID", "es_dorado", "plano",
           "n_cortes", "fuente")]
    print(f"{len(sel)} estudios seleccionados\n")
    print(f"{'etiqueta':<20} {'positivos':>10} {'prevalencia':>12}")
    print("-" * 46)
    cuentas = []
    for c in ET:
        try:
            n = sum(1 for r in sel if r[c] == "1")
        except KeyError:
            continue
        cuentas.append((n, c))
    for n, c in sorted(cuentas):
        print(f"{c:<20} {n:>10} {100*n/len(sel):>11.1f}%")
    if cuentas:
        print(f"\n  factor entre la mas comun y la mas rara: "
              f"{max(cuentas)[0]/max(1,min(cuentas)[0]):.1f}x")
        print("  Ese desbalance es el entregable 6.")

573 estudios seleccionados

etiqueta              positivos  prevalencia
----------------------------------------------
idioma                        0         0.0%
Lateral OA                   65        11.3%
Medial OA                    80        14.0%
Fracture                     81        14.1%
MCL                          88        15.4%
Synovitis                    91        15.9%
Contusion                   117        20.4%
Baker's                     123        21.5%
ACL                         149        26.0%
PF OA                       183        31.9%
Lateral Meniscus            208        36.3%
Effusion                    259        45.2%
Medial Meniscus             331        57.8%

  factor entre la mas comun y la mas rara: 331.0x
  Ese desbalance es el entregable 6.


### La salida: 2.5D, no 3D

Se arman **8 grupos de 3 cortes contiguos** por estudio: 24 cortes que
entran a la red como si fueran imágenes RGB. No es 3D por una razón de
presupuesto —una convolución 3D multiplica el costo— y no es 2D suelto
porque un corte aislado pierde la continuidad anatómica que el radiólogo
usa para seguir un ligamento entre cortes.

# Paso 3 — Las dos arquitecturas (entregable 2)

El enunciado pide **dos**: una CNN desde cero y una con *transfer
learning* sobre ImageNet, las dos con **doce salidas sigmoides
independientes** —no softmax, porque un estudio puede tener varios
hallazgos a la vez—.

In [7]:
h = {}
for n in ("inception_bce_atencion", "resnet50_bce_atencion", "resnet50_promedio"):
    d = leer(f"pregunta1/paso3_modelos/corridas/{n}/historial.json")
    if d:
        h[n] = d
if h:
    print(f"{'corrida':<26} {'params':>9} {'mejor ep':>9} {'AUC val':>9} {'min':>7}")
    print("-" * 64)
    for n, d in h.items():
        print(f"{n:<26} {d.get('parametros_millones', 0):>7.2f} M "
              f"{d.get('mejor_epoca', 0):>9} "
              f"{d.get('auc_validacion', 0):>9.4f} "
              f"{d.get('minutos', 0):>7.1f}")

corrida                       params  mejor ep   AUC val     min
----------------------------------------------------------------
inception_bce_atencion        0.70 M        23    0.6978     2.4
resnet50_bce_atencion        24.06 M         8    0.7240     1.5
resnet50_promedio            23.53 M        11    0.7616     1.8


### El determinismo no es un lujo

Dos corridas con la misma semilla tienen que dar **exactamente** el mismo
resultado, o cualquier ablación posterior mide ruido en vez de medir el
factor que se quiso cambiar.

In [8]:
for arch, nombre in (("determinismo.json", "CON el arreglo"),
                     ("determinismo_sin_arreglo.json", "SIN el arreglo")):
    d = leer(f"pregunta1/paso3_modelos/{arch}")
    if d:
        print(f"{nombre}:")
        for k in ("primera_epoca_distinta", "auc_a", "auc_b",
                  "diferencia_maxima"):
            if k in d:
                print(f"    {k}: {d[k]}")
        print()

CON el arreglo:

SIN el arreglo:



# Paso 4 — Evaluación (entregable 3)

## La regla que sostiene todo el trabajo

El AUC de **validación** se mide contra las etiquetas que fabricó el
extractor: dice *"qué tan bien el modelo reproduce nuestras etiquetas"*.
El AUC de **dorados** se mide contra el radiólogo: dice *"qué tan bien
acierta"*. **Solo el segundo es una medida de calidad**, y es el que se
reporta.

In [9]:
r = leer("pregunta1/paso4_evaluacion/resultados.json")
if r:
    corridas = r if isinstance(r, list) else r.get("corridas", [])
    if isinstance(corridas, dict):
        corridas = [{"nombre": k, **v} for k, v in corridas.items()]
    print(f"{'corrida':<30} {'AUC val':>9} {'AUC oro':>9} {'IC95% oro':>20}")
    print("-" * 72)
    for c in sorted(corridas, key=lambda x: -(x.get("auc_dorados") or
                                              x.get("auc_oro") or 0)):
        nom = c.get("nombre") or c.get("corrida", "?")
        av = c.get("auc_validacion", float("nan"))
        ao = c.get("auc_dorados", c.get("auc_oro", float("nan")))
        ic = c.get("ic95") or c.get("ic95_dorados") or [float("nan")]*2
        print(f"{nom:<30} {av:>9.4f} {ao:>9.4f} "
              f"  [{ic[0]:.3f}, {ic[1]:.3f}]")

corrida                          AUC val   AUC oro            IC95% oro
------------------------------------------------------------------------
resnet50_promedio                 0.7616    0.7085   [0.662, 0.755]
resnet50_promedio_ponderada       0.7582    0.6992   [0.648, 0.749]
resnet50_promedio_focal           0.7449    0.6922   [0.631, 0.752]
resnet50_promedio_coronal         0.7490    0.6502   [0.591, 0.711]
resnet50_bce_atencion             0.7240    0.6461   [0.579, 0.703]
inception_bce_atencion            0.6978    0.6281   [0.569, 0.687]
resnet50_focal                    0.7245    0.6216   [0.571, 0.673]
inception_focal                   0.6513    0.5900   [0.532, 0.642]
resnet50_sin_imagenet             0.6406    0.5623   [0.505, 0.618]


### El resultado que ningún README menciona

Hay un `ensamble.json` en este paso con el **AUC más alto de todo el
repositorio**, y no aparece en ninguna tabla.

In [10]:
e = leer("pregunta1/paso4_evaluacion/ensamble.json")
if e:
    print(f"mejor miembro suelto     : {e['mejor_miembro']:.4f}")
    for k in ("media_probabilidades", "media_rangos"):
        if k in e:
            d = e[k]
            print(f"{k:<25}: {d['auc']:.4f} "
                  f"[{d['ic95'][0]:.4f}, {d['ic95'][1]:.4f}]")
    g = e["media_probabilidades"]["auc"] - e["mejor_miembro"]
    print(f"\nganancia del ensamble: {g:+.4f}")
    print("\n  Es la mejor cifra del repositorio. Que sea PEQUENA es el")
    print("  resultado interesante: ensamblar no arregla una supervision")
    print("  ruidosa, la promedia. El limite no es el modelo.")

mejor miembro suelto     : 0.7217
media_probabilidades     : 0.7269 [0.6771, 0.7724]
media_rangos             : 0.7226 [0.6712, 0.7701]

ganancia del ensamble: +0.0052

  Es la mejor cifra del repositorio. Que sea PEQUENA es el
  resultado interesante: ensamblar no arregla una supervision
  ruidosa, la promedia. El limite no es el modelo.


# Paso 5 — Ablación, Grad-CAM y desbalance (entregables 4, 5 y 6)

## Bootstrap pareado: la mayoría de las diferencias no son reales

Con 56 estudios dorados, una diferencia de AUC menor a ~0.06 no se
distingue del ruido. Medirlo cambia qué conclusiones se pueden escribir.

In [11]:
c = leer("pregunta1/paso5_ablacion_gradcam/comparaciones.json")
if c:
    comps = c if isinstance(c, list) else c.get("comparaciones", [])
    if isinstance(comps, dict):
        comps = [{"nombre": k, **v} for k, v in comps.items()]
    print(f"{'comparacion':<34} {'delta':>8} {'IC95%':>20} {'sig':>5}")
    print("-" * 72)
    reales = 0
    for x in comps:
        nom = x.get("nombre") or x.get("comparacion", "?")
        d = x.get("delta", float("nan"))
        ic = x.get("ic95", [float("nan")]*2)
        s = x.get("significativa", False)
        reales += bool(s)
        print(f"{str(nom)[:33]:<34} {d:>+8.4f} "
              f"[{ic[0]:+.3f},{ic[1]:+.3f}] {'SI' if s else 'no':>5}")
    print(f"\n  reales: {reales} de {len(comps)}")
    print("\n  Las que SI son reales son sobre los DATOS (preentrenamiento,")
    print("  plano de la resonancia), no sobre la arquitectura. Es el")
    print("  mensaje central de esta pregunta.")

  (falta pregunta1/paso5_ablacion_gradcam/comparaciones.json: ese paso todavia no corrio)


## Grad-CAM (entregable 5)

No alcanza con que el modelo acierte: hay que ver **si mira donde debe**.

In [12]:
g = leer("pregunta1/paso5_ablacion_gradcam/figuras/resumen.json")
if g:
    casos = g if isinstance(g, list) else g.get("casos", [])
    print(f"{'hallazgo':<20} {'probabilidad':>13} {'% area caliente':>17}")
    print("-" * 54)
    for x in casos:
        print(f"{x.get('hallazgo','?'):<20} "
              f"{x.get('probabilidad',0):>13.4f} "
              f"{100*x.get('fraccion_caliente', x.get('area',0)):>16.2f}%")
    print(f"\n  {len(casos)} figuras generadas (el enunciado pide al menos 3).")

hallazgo              probabilidad   % area caliente
------------------------------------------------------
Effusion                    0.5764             0.00%
Medial Meniscus             0.4064             0.00%
ACL                         0.8149             0.00%

  3 figuras generadas (el enunciado pide al menos 3).


In [13]:
p = leer("pregunta1/paso5_ablacion_gradcam/patron_gradcam.json")
if p:
    casos = p.get("hallazgos") or p.get("casos") or []
    if isinstance(casos, dict):
        casos = [{"hallazgo": k, **v} for k, v in casos.items()]
    print("Un solo mapa por hallazgo puede ser casualidad. Con 8 casos por")
    print("hallazgo se puede ver si la activacion es CONSISTENTE.\n")
    print(f"{'hallazgo':<20} {'disp':>7} {'borde':>7} {'consistente':>13}")
    print("-" * 52)
    for x in sorted(casos, key=lambda z: z.get("dispersion", 9)):
        print(f"{x.get('hallazgo','?'):<20} "
              f"{x.get('dispersion',0):>7.3f} "
              f"{x.get('casos_en_borde', x.get('borde',0)):>7} "
              f"{str(x.get('consistente','?')):>13}")

Un solo mapa por hallazgo puede ser casualidad. Con 8 casos por
hallazgo se puede ver si la activacion es CONSISTENTE.

hallazgo                disp   borde   consistente
----------------------------------------------------
Lateral Meniscus       0.152       0          True
ACL                    0.163       0          True
Effusion               0.183       0          True
Synovitis              0.184       0         False
MCL                    0.195       0         False
Lateral OA             0.219       0         False
Medial Meniscus        0.247       0         False
Contusion              0.249       0         False
Fracture               0.269       0         False
Medial OA              0.270       0         False
PF OA                  0.302       0         False
Baker's                0.316       0         False


# Resumen de los siete entregables

| # | qué pide | dónde |
|---|---|---|
| 1 | pipeline de etiquetas débiles validado contra los dorados | paso 1 |
| 2 | CNN desde cero + transfer learning, 12 sigmoides | paso 3 |
| 3 | AUC macro + desglose + precisión/recall/F1 a un umbral justificado | paso 4 |
| 4 | ablación de ≥2 factores | paso 5 |
| 5 | Grad-CAM para ≥3 hallazgos | paso 5 |
| 6 | desbalance + ≥1 mitigación evaluada | paso 5 |
| 7 | media página sobre sesgo geográfico/lingüístico | paso 1 + paso 6 |

# Las tres lecciones

**1. El cuello de botella es la supervisión, no la arquitectura.** De las
comparaciones medidas con bootstrap pareado, las únicas reales son sobre
los datos. Cambiar de arquitectura no movió la aguja de forma
distinguible del ruido.

**2. Validación y dorados miden cosas distintas.** Una mide "reproduce
nuestras etiquetas", la otra "acierta". Confundirlas es el error más fácil
de cometer acá.

**3. Un promedio esconde el sesgo.** El extractor falla casi cinco veces
más seguido en griego que en inglés, y eso no aparece en ninguna métrica
agregada.